In [ ]:
import yfinance as yf
import pandas as pd

import sys
from pathlib import Path

project_root = Path.cwd().parents[1]
sys.path.append(str(project_root))

from db.base import sqlite_connection
from db.utils_ohlcv import get_all_tickers
from db.sqlite.ingest_ohlcv import ingest_ohlcv

### Fill Sqlite db:

Initialize sqlite db with long history of ohlcv values. In theory deep learning models benefit hugely from lots of data. We need to be careful as very old data may not be accurate or reliable.

In [ ]:
dfs = []

tickers = get_all_tickers()
for name in tickers:

    ticker = yf.Ticker(name)
    # Date when API has no issues with volume or negative values.
    df = ticker.history(start="2006-01-01")
    df = df.reset_index()

    df = df.drop(columns=["Dividends", "Stock Splits"])
    df["Date"] = (pd.to_datetime(df["Date"]).dt.strftime("%Y-%m-%d"))
    df = df.rename(columns={
        "Date": "date",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume",
    })
    df["ticker"] = name
    df = df[["ticker", "date", "open", "high", "low", "close", "volume"]]

    # Ensure no intraday info enters our database.
    df = df.drop(df.tail(1).index)
    dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True)

with sqlite_connection() as conn:  
    ingest_ohlcv(conn, final_df)

### Fill supabase db

In order to make daily predicitions we need to download ohlcv values from cloud. As averaged or rolling features are used (250 max window) we need to initialize our supabase db with a bit of history.

Please upload the csv in supabase.

In [ ]:
placeholders = ",".join(["?"] * len(tickers))

query = f"""
    SELECT ticker, date, open, high, low, close, volume
    FROM (
        SELECT *,
                ROW_NUMBER() OVER (
                    PARTITION BY ticker
                    ORDER BY date DESC
                ) as rn
        FROM ohlcv
        WHERE ticker IN ({placeholders})
    )
    WHERE rn <= 250
    ORDER BY ticker, date ASC;
"""

with sqlite_connection() as conn:
    df = pd.read_sql_query(query, conn, params=tickers)

df.to_csv(project_root / "data" / "supabase.csv", index=False)